In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# 设置随机种子
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

# 创建序列数据
def create_sequences(X, y, seq_len=24):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:(i + seq_len)])
        y_seq.append(y[i + seq_len])
    return np.array(X_seq), np.array(y_seq)

# sMAPE损失函数
def smape_loss(y_true, y_pred):
    return 2.0 * torch.mean(torch.abs(y_pred - y_true) / (torch.abs(y_true) + torch.abs(y_pred) + 1e-8))

# TCN层定义
class TCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, kernel_size=3, dilation=1, dropout=0.1):
        super(TCNLayer, self).__init__()
        self.conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=output_dim,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=(kernel_size-1)*dilation  # 因果卷积填充
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(output_dim)
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, hidden_dim]
        x = x.transpose(1, 2)  # 转换为 [batch_size, hidden_dim, seq_len]
        x = self.conv(x)
        x = x.transpose(1, 2)  # 转换回 [batch_size, seq_len, output_dim]
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_norm(x)
        return x

# FiLM层实现
class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, condition_dim, hidden_dim=64):
        super(FiLMLayer, self).__init__()
        # 条件网络，从条件特征生成缩放和偏置参数
        self.condition_net = nn.Sequential(
            nn.Linear(condition_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, feature_dim * 2)  # 输出缩放和偏置参数
        )
        self.feature_dim = feature_dim
        
    def forward(self, features, condition):
        """
        Args:
            features: 要调制的特征 [batch_size, seq_len, feature_dim]
            condition: 条件特征 [batch_size, seq_len, condition_dim]
        Returns:
            modulated_features: 调制后的特征 [batch_size, seq_len, feature_dim]
        """
        batch_size, seq_len, feature_dim = features.shape
        condition_dim = condition.shape[-1]
        
        # 将条件特征展平以便处理
        condition_flat = condition.reshape(-1, condition_dim)
        
        # 通过条件网络生成FiLM参数
        film_params = self.condition_net(condition_flat)  # [batch_size*seq_len, feature_dim*2]
        film_params = film_params.reshape(batch_size, seq_len, self.feature_dim * 2)
        
        # 分割缩放和偏置参数
        gamma = film_params[:, :, :self.feature_dim]  # 缩放参数
        beta = film_params[:, :, self.feature_dim:]   # 偏置参数
        
        # 应用FiLM调制: output = gamma * features + beta
        modulated_features = gamma * features + beta
        
        return modulated_features

# 因果多头注意力机制（适用于时间序列预测）
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=8, dropout=0.1):
        super(CausalMultiHeadAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        assert self.head_dim * num_heads == hidden_dim, "hidden_dim must be divisible by num_heads"
        
        # 线性变换矩阵
        self.w_q = nn.Linear(hidden_dim, hidden_dim)
        self.w_k = nn.Linear(hidden_dim, hidden_dim)
        self.w_v = nn.Linear(hidden_dim, hidden_dim)
        self.w_o = nn.Linear(hidden_dim, hidden_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
    def _create_causal_mask(self, seq_len):
        """创建因果掩码，确保不能看到未来信息"""
        # 创建一个下三角矩阵，对角线及以上为1，以下为0
        mask = torch.tril(torch.ones(seq_len, seq_len))
        return mask
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, hidden_dim]
        Returns:
            context_vector: [batch_size, hidden_dim]
            attention_weights: [batch_size, num_heads, seq_len, seq_len]
        """
        batch_size, seq_len, hidden_dim = x.shape
        
        # 线性变换
        Q = self.w_q(x)  # [batch_size, seq_len, hidden_dim]
        K = self.w_k(x)  # [batch_size, seq_len, hidden_dim]
        V = self.w_v(x)  # [batch_size, seq_len, hidden_dim]
        
        # 重塑为多头
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # [batch_size, num_heads, seq_len, head_dim]
        
        # 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        # [batch_size, num_heads, seq_len, seq_len]
        
        # 创建因果掩码并应用到注意力分数上
        causal_mask = self._create_causal_mask(seq_len).to(scores.device)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
        scores = scores.masked_fill(causal_mask == 0, -1e9)
        
        # 应用softmax
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # 应用注意力权重到V
        weighted_V = torch.matmul(attention_weights, V)
        # [batch_size, num_heads, seq_len, head_dim]
        
        # 合并多头
        weighted_V = weighted_V.transpose(1, 2).contiguous().view(
            batch_size, seq_len, hidden_dim
        )
        # [batch_size, seq_len, hidden_dim]
        
        # 输出线性变换
        output = self.w_o(weighted_V)
        output = self.dropout(output)
        
        # 残差连接和层归一化
        output = self.layer_norm(output + x)
        
        # 对于时间序列预测，我们通常只关心最后一个时间步的输出
        context_vector = output[:, -1, :]  # [batch_size, hidden_dim]
        
        return context_vector, attention_weights

# LSTM + 因果多头注意力 模型
class LSTMCausalAttentionModel(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, num_lstm_layers=2, num_heads=8, dropout=0.1):
        super(LSTMCausalAttentionModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_lstm_layers = num_lstm_layers
        
        # LSTM部分
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0,
            bidirectional=False
        )
        
        # 因果多头注意力层
        self.causal_attention = CausalMultiHeadAttention(
            hidden_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout
        )
        
        # 层归一化
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # 输出层
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        
        # LSTM处理
        lstm_out, (hidden, cell) = self.lstm(x)
        # lstm_out shape: [batch_size, seq_len, hidden_dim]
        
        # 层归一化
        lstm_out = self.layer_norm(lstm_out)
        
        # 使用因果多头注意力机制聚合序列信息
        context_vector, attention_weights = self.causal_attention(lstm_out)
        # context_vector shape: [batch_size, hidden_dim]
        
        # 最终输出
        output = self.output_layer(context_vector)
        # output shape: [batch_size, 1]
        
        return output

# 单特征处理模型（简化版）
class SingleFeatureModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, num_layers=2, dropout=0.1):
        super(SingleFeatureModel, self).__init__()
        
        # 简单的多层感知机处理单特征
        layers = []
        current_dim = input_dim
        for i in range(num_layers):
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.LayerNorm(hidden_dim)
            ])
            current_dim = hidden_dim
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, 1]
        batch_size, seq_len, input_dim = x.shape
        
        # 展平处理
        x_flat = x.reshape(-1, input_dim)
        # 通过网络
        out_flat = self.network(x_flat)
        # 恢复形状
        out = out_flat.reshape(batch_size, seq_len, -1)
        
        return out

# 双模型 + FiLM + TCN 融合模型（使用LSTM+因果多头注意力）
class DualModelFiLMTCNFusion(nn.Module):
    def __init__(self, input_dim1=3, input_dim2=1, hidden_dim=64, num_lstm_layers=2, 
                 tcn_layers=2, num_heads=8, dropout=0.1, use_causal_attention=True):
        super(DualModelFiLMTCNFusion, self).__init__()
        
        # 第一个模型（3个输入特征）- 多特征模型（使用LSTM+因果多头注意力）
        self.multi_feature_model = LSTMCausalAttentionModel(
            input_dim=input_dim1,
            hidden_dim=hidden_dim,
            num_lstm_layers=num_lstm_layers,
            num_heads=num_heads,
            dropout=dropout
        )
        
        # 第二个模型（1个输入特征）- 单特征模型
        self.single_feature_model = SingleFeatureModel(
            input_dim=input_dim2,
            hidden_dim=hidden_dim,
            num_layers=2,
            dropout=dropout
        )
        
        # FiLM融合层
        self.film_layer = FiLMLayer(
            feature_dim=hidden_dim,  # 要调制的特征维度
            condition_dim=hidden_dim,  # 条件特征维度
            hidden_dim=hidden_dim // 2
        )
        
        # TCN部分（TCN本身是因果的）
        self.tcn_layers = nn.ModuleList()
        tcn_input_dim = hidden_dim  # FiLM融合后维度不变
        for i in range(tcn_layers):
            dilation = 2 ** i  # 指数增长的膨胀系数
            self.tcn_layers.append(
                TCNLayer(tcn_input_dim, tcn_input_dim, kernel_size=3, dilation=dilation, dropout=dropout)
            )
        
        # 全局因果多头注意力层（在TCN之后应用）
        self.global_causal_attention = CausalMultiHeadAttention(
            hidden_dim=tcn_input_dim,
            num_heads=num_heads,
            dropout=dropout
        ) if use_causal_attention else None
        
        # 输出层
        self.output_layer = nn.Sequential(
            nn.Linear(tcn_input_dim, tcn_input_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(tcn_input_dim // 2, 1)
        )
        
    def forward(self, x1, x2):
        """
        Args:
            x1: 多特征输入 [batch_size, seq_len, 3]
            x2: 单特征输入 [batch_size, seq_len, 1]
        """
        # 多特征模型处理 - 获取LSTM的输出
        lstm_out, (hidden, cell) = self.multi_feature_model.lstm(x1)
        multi_feature_out = self.multi_feature_model.layer_norm(lstm_out)
        # multi_feature_out shape: [batch_size, seq_len, hidden_dim]
        
        # 单特征模型处理
        single_feature_out = self.single_feature_model(x2)  # [batch_size, seq_len, hidden_dim]
        
        # FiLM融合：使用单特征作为条件来调制多特征
        # 这里我们将多特征作为主要特征，单特征作为条件
        film_fused = self.film_layer(multi_feature_out, single_feature_out)
        # film_fused shape: [batch_size, seq_len, hidden_dim]
        
        # 可选：也可以尝试残差连接
        fused_output = film_fused + multi_feature_out  # 残差连接
        
        # TCN处理（TCN是因果的，不会看到未来信息）
        tcn_out = fused_output
        for tcn_layer in self.tcn_layers:
            tcn_out = tcn_layer(tcn_out)
        # tcn_out shape: [batch_size, seq_len, hidden_dim]
        
        # 应用全局因果注意力或取最后一个时间步
        if self.global_causal_attention is not None:
            # 使用因果注意力机制聚合序列信息
            context_vector, attention_weights = self.global_causal_attention(tcn_out)
            # context_vector shape: [batch_size, hidden_dim]
        else:
            # 不使用注意力，直接取最后一个时间步
            context_vector = tcn_out[:, -1, :]  # [batch_size, hidden_dim]
        
        # 最终输出
        output = self.output_layer(context_vector)  # [batch_size, 1]
        
        return output

def main():
    set_seed(42)  # 固定随机种子

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # 1. 读取Excel数据
    df = pd.read_excel("C:\\Users\\浩琦\\xwechat_files\\wxid_g7jgiynd27oh22_97e4\\msg\\file\\2025-10\\5.xlsx")  # 修改为你的Excel文件名
    X1 = df.iloc[:, 1:4].values  # 第2-4列作为第一个模型的输入
    X2 = df.iloc[:, 1:2].values  # 第2列作为第二个模型的输入
    y = df.iloc[:, 5].values     # 第5列作为标签

    # 2. 数据归一化
    scaler_X1 = MinMaxScaler()
    scaler_X2 = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X1 = scaler_X1.fit_transform(X1)
    X2 = scaler_X2.fit_transform(X2)
    y = scaler_y.fit_transform(y.reshape(-1, 1))

    # 3. 构造24步序列
    seq_len = 24
    X1_seq, y_seq = create_sequences(X1, y, seq_len=seq_len)
    X2_seq, _ = create_sequences(X2, y, seq_len=seq_len)

    # 转换为Tensor
    X1_tensor = torch.tensor(X1_seq, dtype=torch.float32)  # [N, 24, 3]
    X2_tensor = torch.tensor(X2_seq, dtype=torch.float32)  # [N, 24, 1]
    y_tensor = torch.tensor(y_seq, dtype=torch.float32)    # [N, 1]

    # 创建包含两个输入的数据集
    dataset = TensorDataset(X1_tensor, X2_tensor, y_tensor)

    # 划分训练集/测试集（加上生成器保证可复现）
    train_size = int(0.7 * len(dataset))
    test_size = len(dataset) - train_size
    generator = torch.Generator().manual_seed(42)
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=generator)
    test_loader = DataLoader(test_dataset, batch_size=32)

    # 4. 定义双模型FiLM融合模型（使用LSTM+因果多头注意力）
    model = DualModelFiLMTCNFusion(
        input_dim1=3,
        input_dim2=1,
        hidden_dim=128,
        num_lstm_layers=2,
        tcn_layers=2,
        num_heads=8,  # 8个头
        dropout=0.1,
        use_causal_attention=True  # 使用因果注意力
    ).to(device)
    
    print(f"双模型FiLM融合模型参数数量: {sum(p.numel() for p in model.parameters()):,}")
    print(f"使用的RNN类型: LSTM")
    print(f"使用因果多头注意力: True, 头数: 8")
    
    criterion = nn.L1Loss()  # MAE 损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=20, factor=0.5)

    # 5. 训练
    epochs = 500
    best_mae = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for xb1, xb2, yb in train_loader:
            xb1, xb2, yb = xb1.to(device), xb2.to(device), yb.to(device)

            pred = model(xb1, xb2)
            loss = criterion(pred, yb)  # MAE作为loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
            optimizer.step()

            train_loss += loss.item() * xb1.size(0)

        train_loss /= len(train_loader.dataset)

        # 6. 测试
        model.eval()
        mae_total, mse_total, smape_total, n_samples = 0.0, 0.0, 0.0, 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for xb1, xb2, yb in test_loader:
                xb1, xb2, yb = xb1.to(device), xb2.to(device), yb.to(device)
                pred = model(xb1, xb2)

                # 逆归一化
                pred_denorm = scaler_y.inverse_transform(pred.cpu().numpy())
                yb_denorm = scaler_y.inverse_transform(yb.cpu().numpy())

                pred_t = torch.tensor(pred_denorm, dtype=torch.float32)
                yb_t = torch.tensor(yb_denorm, dtype=torch.float32)

                # 计算 MAE, RMSE, sMAPE
                mae_total += torch.sum(torch.abs(pred_t - yb_t)).item()
                mse_total += torch.sum((pred_t - yb_t) ** 2).item()
                smape_total += smape_loss(yb_t, pred_t).item() * len(xb1)
                n_samples += len(xb1)
                
                # 收集所有预测值和真实值用于计算R²
                all_preds.extend(pred_denorm.flatten())
                all_targets.extend(yb_denorm.flatten())

        mae = mae_total / n_samples
        rmse = np.sqrt(mse_total / n_samples)
        smape = smape_total / n_samples
        
        # 计算R平方
        r_squared = r2_score(all_targets, all_preds)
        
        # 学习率调度
        scheduler.step(mae)
        
        # 保存最佳模型
        if mae < best_mae:
            best_mae = mae
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_mae': best_mae,
            }, 'best_lstm_causal_attention_film_tcn_fusion.pth')
        
        # 每个epoch都打印结果
        print(f"Epoch {epoch+1}/{epochs}, TrainLoss(MAE)={train_loss:.6f}, "
              f"Test MAE={mae:.6f}, Test RMSE={rmse:.6f}, Test sMAPE={smape:.4f}, "
              f"R²={r_squared:.4f}, LR={optimizer.param_groups[0]['lr']:.6f}")

    print(f"\n训练完成！最佳测试MAE: {best_mae:.6f}")

if __name__ == "__main__":
    main()

Using device: cuda
双模型FiLM融合模型参数数量: 490,946
使用的RNN类型: LSTM
使用因果多头注意力: True, 头数: 8
Epoch 1/500, TrainLoss(MAE)=0.096694, Test MAE=10.300633, Test RMSE=12.455329, Test sMAPE=0.3081, R²=0.9318, LR=0.001000
Epoch 2/500, TrainLoss(MAE)=0.063898, Test MAE=5.465111, Test RMSE=7.010785, Test sMAPE=0.2108, R²=0.9784, LR=0.001000
Epoch 3/500, TrainLoss(MAE)=0.058632, Test MAE=9.160269, Test RMSE=10.673363, Test sMAPE=0.3164, R²=0.9499, LR=0.001000
Epoch 4/500, TrainLoss(MAE)=0.052878, Test MAE=6.273575, Test RMSE=7.618161, Test sMAPE=0.2738, R²=0.9745, LR=0.001000
Epoch 5/500, TrainLoss(MAE)=0.051503, Test MAE=8.292002, Test RMSE=10.267149, Test sMAPE=0.2944, R²=0.9537, LR=0.001000
Epoch 6/500, TrainLoss(MAE)=0.052518, Test MAE=5.671466, Test RMSE=7.065587, Test sMAPE=0.2132, R²=0.9781, LR=0.001000
Epoch 7/500, TrainLoss(MAE)=0.051830, Test MAE=5.272050, Test RMSE=6.978089, Test sMAPE=0.2021, R²=0.9786, LR=0.001000
Epoch 8/500, TrainLoss(MAE)=0.046219, Test MAE=8.939718, Test RMSE=10.892224, Tes

KeyboardInterrupt: 